Notwendige Imports (Standardpakete in Python - keine 3rd Party)


In [1]:
import time
import requests
from urllib.parse import quote

Init-Request - holt die IDs der Ressourcen und erstellt eine Liste aller Ressourcen


In [2]:
url = "https://fcs.text-plus.org/rest/init"
resp = requests.get(url)
resp.raise_for_status()

data = resp.json()
resources_id = [item["id"] for item in data.get("resources", [])]
resources_str = "resourceIds[]=" + "&resourceIds[]=".join(resources_id)

resources = {}

for resource in data.get("resources", []):
  resource_id = resource.get("id")
  if not resource_id:
    continue

  resources[resource_id] = {
    "Institution": (resource.get("endpointInstitution") or {}).get("name"),
    "URL": resource.get("landingPage"),
    "Name": (resource.get("title") or {}).get("de"),
  }


Konfiguration - Einstellungen für die Suche


In [10]:
queryType = "lex"
language = "mul" # multilingual
numberOfResults = 100

queryPostfix = f"queryType={queryType}&language={language}&numberOfResults={numberOfResults}&{resources_str}"

headers = {
  "accept": "application/json, text/plain, */*",
  "content-type": "application/x-www-form-urlencoded",
}

Eigentlicher Query


In [20]:
query = "lemma = Berg*"

Bereitet Body für die Suche vor - Query + Konfiguration => Encoder


In [21]:
body = f"query={query}&{queryPostfix}"
body = quote(body, safe="&=")

Eigentliche Suche


In [22]:
resp = requests.post("https://fcs.text-plus.org/rest/search", data=body, headers=headers)
resp.raise_for_status()
search_id = resp.text

Rufe so lange /metaonly auf, bis alle Ressourcen Ergebnisse haben


In [25]:
search_id = search_id.strip('"')
meta_url = f"https://fcs.text-plus.org/rest/search/{search_id}/metaonly"

while True:
  resp = requests.get(meta_url)
  resp.raise_for_status()
  meta = resp.json()
  if meta.get("inProgress") == 0:
    break
  time.sleep(0.2)

results_by_id = {item["id"]: item["numberOfRecords"] for item in meta.get("results", [])}

Gebe für jede Ressource die Anzahl der gefundenen Einträge aus


In [26]:
for resource_id, count in results_by_id.items():
  if count > 0:
    name = resources.get(resource_id, {}).get("Name", resource_id)
    print(f"{name}: {count}")

elexiko - Online-Wörterbuch zur deutschen Gegenwartssprache: 268
Digitales Wörterbuch der deutschen Sprache: 236
Etymologisches Wörterbuch des Deutschen: 10
Deutsches Wörterbuch von Jacob Grimm und Wilhelm Grimm / Neubearbeitung (A-F): 66
Deutsches Wörterbuch von Jacob Grimm und Wilhelm Grimm: 567
Mittelhochdeutsches Handwörterbuch von Matthias Lexer: 11
Nachträge zum Mittelhochdeutschen Handwörterbuch von Matthias Lexer: 5
Mittelhochdeutsches Wörterbuch von Benecke, Müller, Zarncke: 4
Findebuch zum mittelhochdeutschen Wortschatz: 3
Mittelniederdeutsches Handwörterbuch von August Lübben und Christoph Walther: 6
Mittelniederdeutsches Wörterbuch: 33
Goethe-Wörterbuch: 295
Grammatisch-Kritisches Wörterbuch der Hochdeutschen Mundart (Ausgabe letzter Hand, Leipzig 1793–1801): 283
Wörterbuch der elsässischen Mundarten: 13
Wörterbuch der deutsch-lothringischen Mundarten: 1
Pfälzisches Wörterbuch: 41
Rheinisches Wörterbuch: 84
Nachträge zum Rheinischen Wörterbuch: 27
Mecklenburgisches Wörterbu

Lade alle verfügbaren Ergebnisse herunter


In [28]:
search_results = {}
for resource_id, count in results_by_id.items():
  if count > 0:
    resource_url = f"https://fcs.text-plus.org/rest/search/{search_id}?resourceId={quote(resource_id, safe='')}"
    resp = requests.get(resource_url)
    resp.raise_for_status()
    search_results[resource_id] = resp.json()

Gebe den ersten Eintrag aus search_results zurück


In [29]:
first_search_result = next(iter(search_results.items()))
first_search_result

('https://www.owid.de/api/fcs/#https://doi.org/10.14618/wb-elex',
 {'inProgress': 0,
  'results': [{'resourceHandle': 'https://doi.org/10.14618/wb-elex',
    'endpointUrl': 'https://www.owid.de/api/fcs/',
    'inProgress': False,
    'cancelled': False,
    'nextRecordPosition': 11,
    'numberOfRecords': 268,
    'numberOfRecordsLoaded': 10,
    'exception': None,
    'diagnostics': [],
    'requestUrl': 'https://www.owid.de/api/fcs/?query=lemma+%3D+Berg*&queryType=lex&startRecord=1&maximumRecords=10&recordSchema=http%3A%2F%2Fclarin.eu%2Ffcs%2Fresource&x-fcs-context=https%3A%2F%2Fdoi.org%2F10.14618%2Fwb-elex',
    'resource': {'endpointInstitution': {'name': 'Leibniz-Institut für Deutsche Sprache',
      'link': None,
      'endpoints': [{'url': 'https://www.owid.de/api/fcs/',
        'protocol': 'VERSION_2',
        'searchCapabilities': ['BASIC_SEARCH', 'LEX_SEARCH']}],
      'sideloaded': True},
     'endpoint': {'url': 'https://www.owid.de/api/fcs/',
      'protocol': 'VERSION_2',